# 51 · Độ phân giải dưới **ML-Decoder** — `49` hỏi bằng dụng cụ mù

## Vì sao `49` không trả lời được

`49` probe tuyến tính trên `mean(patch)`. Ba con số của nó tuân theo **đúng một luật**:

```
crop518 - lb518   +0.0082   doi NOI DUNG khung (bo ngoai vi + bo dem)   -> trung binh DOI
nosq518 - lb518   +0.0049   doi NOI DUNG khung (bo dem den)             -> trung binh DOI
lb672   - lb518   -0.0010   GIU noi dung, chi LAY MAU MIN HON           -> trung binh KHONG doi
```

> `mean(patch)` phản ứng với **cái gì nằm trong khung**, không phản ứng với **khung được lấy mẫu
> mịn tới đâu**.

Đó là **tính chất toán học**: trung bình của `2,304` mẫu và của `1,369` mẫu **cùng một trường** thì
xấp xỉ bằng nhau. Một tổn thương chiếm `f` phần diện tích chiếm `1369·f` token ở 518 và `2304·f`
token ở 672 — **tỉ lệ không đổi** ⇒ đóng góp vào trung bình không đổi.

⇒ `−0.0010` là con số dụng cụ ấy sẽ trả về **bất kể sự thật là gì**. Công suất ~0.

## Notebook này đổi dụng cụ, không đổi câu hỏi

ML-Decoder cho 40 query lớp **attend trên toàn bộ token**. `37` đo nó hơn phép trung bình
`+0.0387` — lớn gấp 39 lần thứ ta đang cố đọc. Với đầu đọc này:

```
token DEM DEN      attention tu ha trong so   -> phan `+0.0049` cua 49 PHAI bien mat
token NGOAI VI     attention dung KHI CAN     -> mat FOV co the DAT HON o 49
DO PHAN GIAI       attention KHONG the tao ra do phan giai chua duoc lay mau -> phan nay SONG
```

## Ba arm — ô 2×2 khuyết một ô

```
                0.812 mm/px      0.626 mm/px
FOV  71.5%          —            crop518
FOV 100.0%       lb518            lb672
```

| tương phản | đổi cái gì | đọc được gì |
|---|---|---|
| `lb672 − lb518` | **chỉ** `mm/px` (FOV 100%, đệm 18% giữ nguyên) | **độ phân giải thuần** ← câu hỏi chính |
| `crop518 − lb518` | `mm/px` **+** FOV **+** đệm | "mua độ phân giải bằng FOV" có lãi không |
| `crop518 − lb672` | **chỉ** FOV (cùng `0.626 mm/px`) | FOV đáng giá bao nhiêu ở độ phân giải cao |

`crop518` đáng chú ý về **chi phí**: nó cho đúng `0.626 mm/px` của `lb672` mà **cùng 1,369 token**
⇒ **cùng FLOPs với `lb518`**. Nó trả bằng FOV thay vì bằng compute.

## Backbone **pretrain**, không phải `ft`

`38a` fine-tune trên `lb518`. Đưa `crop518`/`lb672` vào backbone đó là **lệch miền train/test**:
arm thua sẽ không phân biệt được *"khung dở"* với *"backbone chưa từng thấy khung này"*.

Backbone pretrain **không thiên vị arm nào**. Đổi lại mAP tuyệt đối ~0.21–0.22 chứ không phải
`0.2504` — **không sao**, ta đọc **hiệu số ghép cặp**, không đọc số tuyệt đối.

Nếu một arm thắng ở đây thì bước tiếp là fine-tune `38a` trên chính arm đó (~6 h) để thu về ở mức `ft`.

## Ghép cặp

Một mẫu trả về **cả ba ảnh** của **cùng một `dicom_id`**, ba đầu đọc học trong **một** vòng lặp,
**cùng** thứ tự dữ liệu, **cùng** nhãn, **cùng** seed. Tham số ba arm **rời nhau** ⇒ theo lập luận
① của `AMENDMENTS`, huấn luyện chung một vòng lặp là **tương đương toán học tuyệt đối** với chạy
riêng. Hiệu `arm − lb518` vì thế có phương sai thấp hơn hẳn số tuyệt đối.

**Không có EHR trong notebook này.** `G_ehr = +0.007` đã lặp lại ở 4 notebook độc lập và là số hạng
**cộng thêm**; để nó vào đây chỉ thêm arm và thêm phương sai.

## C0 · Config

In [ ]:
import os, json, math, time, random, shutil, tarfile, io as _io, re, glob
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

try:
    from google.colab import drive
    drive.mount('/content/drive'); DATA_DIR = '/content/drive/MyDrive/MIMIC_CXR/planB_out'
except Exception:
    DATA_DIR = './planB_out'

OUT_DIR = os.path.join(DATA_DIR, 'cxrlt_2024')
EMB_DIR = os.path.join(OUT_DIR, 'embeddings')
RUN_TAG = 'r1_pre'
RES51   = os.path.join(OUT_DIR, 'stage4_nb51', RUN_TAG)
CKPT    = os.path.join(RES51, 'ckpt')

# KHONG bao gio va `builtins.open` — lambda giai ten `_open` nhu bien TOAN CUC luc goi, chay lai
# cell lan 2 la de quy vo han; va worker DataLoader goi open(fd) trong `_close_stdin` luc fork.
def ensure_res():
    '''Goi TRUOC MOI cum ghi. Drive FUSE danh roi thu muc RONG: mot epoch ~90 phut khong ghi gi
       la du de thu muc bien mat va `np.save` bao FileNotFoundError (da dinh o 43, 38c, 38d).'''
    os.makedirs(RES51, exist_ok=True); os.makedirs(CKPT, exist_ok=True)
    return RES51
ensure_res()
json.dump({'run_tag': RUN_TAG, 'notebook': '51'}, open(os.path.join(RES51, '_run_info.json'), 'w'))
open(os.path.join(CKPT, '_keep'), 'w').write('keep')

# ---------------- BACKBONE ----------------
BB_MODE = 'pre'      # 'pre' = RAD-DINO pretrain (KHUYEN NGHI: khong thien vi arm nao)
                     # 'ft'  = 38a/a2_E3 — CHI dung neu chap nhan lech mien: no fine-tune tren
                     #         `lb518`, nen ket qua AM cua crop518/lb672 se KHONG doc duoc.
BB_RUN  = 'a2_E3'

# ---------------- BA ARM ANH ----------------
# `mm_px` va `fov` lay tu 31/49; `pad` la ty le khung la dem den (33 C2 do letterbox ~18%).
ARM_IMG = {
    'lb518':   dict(local='/content/cxrlt_518',     size=518, mm_px=0.812, fov=1.000, pad=0.18,
                    src='lb',   note='letterbox 1022 -> 518'),
    'crop518': dict(local='/content/cxrlt_crop518', size=518, mm_px=0.626, fov=0.715, pad=0.00,
                    src='crop', note='shard 559 -> CenterCrop(518)'),
    'lb672':   dict(local='/content/cxrlt_672',     size=672, mm_px=0.626, fov=1.000, pad=0.18,
                    src='lb',   note='letterbox 1022 -> 672'),
}
ARMS = list(ARM_IMG)

# nguon shard — LAY TU 31 C0, khong dung lai tu tri nho
SRC_DIR = {
    'lb':   [(os.path.join(DATA_DIR, 'cxr_1022_letterbox_cxrlt'), 'lb_')],
    'crop': [(os.path.join(DATA_DIR, 'cxr_560_shards'),       'shard_'),
             (os.path.join(DATA_DIR, 'cxr_560_shards_cxrlt'), 'cxrlt_')],
}

# moc `mean(patch)` cua 49 — de C5 kiem xem thanh phan PHA LOANG co bien mat khong
MOC49 = {'lb518': 0.2044, 'crop518': 0.2126, 'lb672': 0.2034}

SEEDS  = (42,)          # 1 seed ~13-17 h. Them 43 neu |d| sat bien do (resume o muc epoch nen
                        # chi can doi tuple roi chay lai C4).
SEED   = 42
EPOCHS = 10             # nhu 45. Chon `E` bang dong `LAT CHON`, KHONG bang dong `VAL`.
BATCH, ACCUM = 64, 4    # BA anh moi mau (2x518 + 1x672). OOM thi ha 32x8 — EFF giu 256.
LR, WD, WARMUP, CLIP = 1e-3, 1e-2, 0.05, 5.0
W_FRONTAL, W_LATERAL = 0.7, 0.3
NW = min(8, (os.cpu_count() or 2))
D_MLD = 512

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
assert dev == 'cuda', 'KHONG thay GPU -> Runtime > Change runtime type > GPU roi chay lai tu C0.'
AMP = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(f'thiet bi: {torch.cuda.get_device_name(0)} | '
      f'VRAM {torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB | AMP {AMP}')

for a, c in ARM_IMG.items():
    for d, _ in SRC_DIR[c['src']]:
        assert os.path.isdir(d), f'arm {a}: khong thay shard {d} -> chay 28b/28c truoc'
# --- cong 0: ba arm phai co ba thu muc anh RIENG. Hai arm tro cung mot thu muc thi moi hieu
#     so deu bang 0 va notebook van chay tron tru — loi im lang, phai chan ngay o day. ---
_lc = [c['local'] for c in ARM_IMG.values()]
assert len(set(_lc)) == len(_lc), f'trung thu muc anh giua cac arm: {_lc}'

INV   = json.load(open(os.path.join(OUT_DIR, 'cxrlt_inventory.json')))
LAB40 = INV['labels_task1']; C = len(LAB40)
TIER  = {t: np.array([LAB40.index(l) for l in INV['tier'][t]]) for t in ('head','medium','tail')}

ORDER = pd.read_parquet(os.path.join(EMB_DIR, 'emb_index.parquet'))
Y = (pd.read_parquet(os.path.join(OUT_DIR, 'cxrlt_index.parquet'), columns=['dicom_id'] + LAB40)
       .set_index('dicom_id').loc[ORDER.dicom_id].reset_index())
assert (Y.dicom_id.to_numpy() == ORDER.dicom_id.to_numpy()).all()
YM = Y[LAB40].to_numpy('float32')

P_VIEW = os.path.join(OUT_DIR, 'cxrlt_view_resolved.parquet')
V_ = ((pd.read_parquet(P_VIEW)[['dicom_id','view_group','use']]
         .set_index('dicom_id').loc[ORDER.dicom_id].reset_index()) if os.path.exists(P_VIEW)
      else ORDER[['dicom_id','view_group']].assign(use=lambda d: d.view_group != 'unknown'))

SPLIT = ORDER.split.to_numpy(); SUB = ORDER.subject_id.to_numpy()
TR, VA = SPLIT == 'train', SPLIT == 'val'
_rs = np.random.default_rng(SEED); _subs = np.unique(SUB[TR])
_sel = _rs.choice(_subs, size=int(round(0.15 * len(_subs))), replace=False)
SL = TR & np.isin(SUB, _sel); TF_ = TR & ~SL        # CUNG cach cat nhu 35/37/38*/44*/45
assert not (set(SUB[SL]) & set(SUB[TF_])) and not (set(SUB[TR]) & set(SUB[VA]))
I_TF, I_SL, I_VA = np.where(TF_)[0], np.where(SL)[0], np.where(VA)[0]

print(f'{len(ORDER):,} anh | khop {len(I_TF):,} | chon {len(I_SL):,} | val {len(I_VA):,}')
print(f'backbone `{BB_MODE}` | seeds {SEEDS} | E={EPOCHS} | BATCH {BATCH}x{ACCUM} | ghi vao {RES51}')
print(f'\n{"arm":<9} {"px":>5} {"token":>6} {"mm/px":>7} {"FOV":>7} {"dem":>5}  nguon')
for a, c in ARM_IMG.items():
    print(f'{a:<9} {c["size"]:>5} {(c["size"]//14)**2:>6,} {c["mm_px"]:>7.3f} '
          f'{c["fov"]:>6.1%} {c["pad"]:>5.0%}  {c["note"]}')
print('\nDIA LOCAL: ba bo anh ~20+20+35 = ~75 GB. Kiem `df -h /content` truoc khi chay C1.')

Mounted at /content/drive
thiet bi: NVIDIA RTX PRO 6000 Blackwell Server Edition | VRAM 102 GB | AMP torch.bfloat16
377,110 anh | khop 219,785 | chon 39,086 | val 39,293
backbone `pre` | seeds (42,) | E=10 | BATCH 64x4 | ghi vao /content/drive/MyDrive/MIMIC_CXR/planB_out/cxrlt_2024/stage4_nb51/r1_pre

arm          px  token   mm/px     FOV   dem  nguon
lb518       518  1,369   0.812 100.0%   18%  letterbox 1022 -> 518
crop518     518  1,369   0.626  71.5%    0%  shard 559 -> CenterCrop(518)
lb672       672  2,304   0.626 100.0%   18%  letterbox 1022 -> 672

DIA LOCAL: ba bo anh ~20+20+35 = ~75 GB. Kiem `df -h /content` truoc khi chay C1.


## C1 · Stage **ba** bộ ảnh + backbone đông cứng

Mỗi bộ có `_done.json` riêng, checkpoint **từng tar** ⇒ đứt thì chạy lại cell này, nó tiếp từ chỗ dở.
`lb518` (`45`) và `lb672` (`49`) có thể đã sẵn ở `/content` nếu chưa restart session.

Đọc **bulk** cả tar vào RAM rồi mở `tarfile` trên `BytesIO`; lặp `for m in tar` thẳng trên mount
FUSE sẽ **treo cứng** (đã dính ở `33`).

In [ ]:
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
from transformers import AutoModel, AutoImageProcessor
from torchvision import transforms as T

def stage(arm):
    '''Trich anh cua mot arm xuong dia local. Tra ve so anh co tren dia.'''
    cfg = ARM_IMG[arm]; loc, size, src = cfg['local'], cfg['size'], cfg['src']
    os.makedirs(loc, exist_ok=True)
    n = len([f for f in os.listdir(loc) if f.endswith('.jpg')])
    print(f'  {arm:<9} local {n:>7,} / {len(ORDER):,}', flush=True)
    if n >= len(ORDER) * 0.999: return n
    TARS = []
    for d, pref in SRC_DIR[src]:
        TARS += sorted(os.path.join(d, f) for f in os.listdir(d)
                       if f.endswith('.tar') and f.startswith(pref))
    DONE = os.path.join(loc, '_done.json')
    done = set(json.load(open(DONE))) if os.path.exists(DONE) else set()
    def _one(pl):
        name, blob = pl
        dst = os.path.join(loc, os.path.basename(name))
        if os.path.exists(dst): return
        im = Image.open(_io.BytesIO(blob)).convert('L')
        if src == 'crop':                      # 559 -> CenterCrop(518), tai lap dung 31 C3
            w, h = im.size
            assert w >= size and h >= size, f'{arm}: anh {w}x{h} nho hon {size}, khong crop duoc'
            l, t = (w - size) // 2, (h - size) // 2
            im = im.crop((l, t, l + size, t + size))
        elif im.size != (size, size):          # letterbox 1022 -> size
            im = im.resize((size, size), Image.BICUBIC)
        im.save(dst, 'JPEG', quality=95)
    t0 = time.time()
    todo = [t for t in TARS if os.path.basename(t) not in done]
    with ThreadPoolExecutor(8) as pool:
        for tp in tqdm(todo, desc=f'stage {arm}', unit='tar'):
            with open(tp, 'rb') as f: buf = f.read()      # BULK, khong lap tren mount
            with tarfile.open(fileobj=_io.BytesIO(buf)) as tar:
                jobs = [(m.name, tar.extractfile(m).read())
                        for m in tar.getmembers() if m.name.endswith('.jpg')]
            list(pool.map(_one, jobs)); del buf, jobs
            done.add(os.path.basename(tp)); json.dump(sorted(done), open(DONE, 'w'))
    n = len([f for f in os.listdir(loc) if f.endswith('.jpg')])
    print(f'  {arm:<9} stage xong {n:,} anh | {(time.time()-t0)/60:.1f} phut', flush=True)
    return n

print('stage ba bo anh:')
for a in ARMS:
    _n = stage(a)
    assert _n >= len(ORDER) * 0.999, f'{a}: thieu {len(ORDER)-_n:,} anh'

proc = AutoImageProcessor.from_pretrained('microsoft/rad-dino')
MEAN, STD = proc.image_mean[0], proc.image_std[0]
# Augmentation TUONG DOI giong nhau moi arm: cung `scale`/`ratio`/goc xoay, chi khac kich thuoc ra.
def tfs(size):
    return (T.Compose([T.RandomResizedCrop(size, scale=(0.85, 1.0), ratio=(0.95, 1.05),
                                           interpolation=T.InterpolationMode.BICUBIC),
                       T.RandomRotation(7), T.ToTensor(), T.Normalize([MEAN]*3, [STD]*3)]),
            T.Compose([T.ToTensor(), T.Normalize([MEAN]*3, [STD]*3)]))
TF_TR, TF_EV = {}, {}
for a in ARMS: TF_TR[a], TF_EV[a] = tfs(ARM_IMG[a]['size'])

# --------- backbone ---------
backbone = AutoModel.from_pretrained('microsoft/rad-dino')
_sd0 = {k: v.clone() for k, v in AutoModel.from_pretrained('microsoft/rad-dino').state_dict().items()}
if BB_MODE == 'ft':
    BB_SRC = os.path.join(OUT_DIR, 'stage4_nb38a', BB_RUN, 'backbone_ft.pt')
    assert os.path.exists(BB_SRC), f'khong thay {BB_SRC}'
    backbone.load_state_dict(torch.load(BB_SRC, map_location='cpu', weights_only=False)['bb'],
                             strict=True)
with torch.no_grad():
    _d = max((_sd0[k] - v).abs().max().item() for k, v in backbone.state_dict().items())
del _sd0
# --- cong 1: backbone PHAI dung che do da khai bao ---
if BB_MODE == 'pre':
    assert _d < 1e-6, (f'BB_MODE="pre" nhung trong so LECH {_d:.2e} so voi RAD-DINO pretrain. '
                       'Ket qua se thien vi arm ma backbone da duoc fine-tune tren do.')
else:
    assert _d > 1e-6, f'BB_MODE="ft" nhung trong so GIONG HET pretrain (lech {_d:.1e})'
backbone = backbone.to(dev).eval()
for p in backbone.parameters(): p.requires_grad_(False)
D_CXR, PATCH = backbone.config.hidden_size, backbone.config.patch_size
print(f'\ncong 1: backbone `{BB_MODE}` OK (lech so voi pretrain {_d:.3g})')

class DS(Dataset):
    '''Mot mau -> BA anh cua CUNG mot dicom_id. Ghep cap tuyet doi: cung nhan, cung thu tu.'''
    def __init__(s, idx, train): s.idx, s.tf = np.asarray(idx), (TF_TR if train else TF_EV)
    def __len__(s): return len(s.idx)
    def __getitem__(s, k):
        i = int(s.idx[k]); dic = ORDER.dicom_id.iloc[i]
        out = {}
        for a in ARMS:
            im = Image.open(os.path.join(ARM_IMG[a]['local'], f'{dic}.jpg')).convert('L')
            out['im_' + a] = s.tf[a](Image.merge('RGB', (im, im, im)))
        out.update(y=torch.from_numpy(YM[i]), row=i, pos=k)
        return out

def loader(idx, train, shuffle, bs, nw=NW, drop_last=None):
    return DataLoader(DS(idx, train), batch_size=bs, shuffle=shuffle, num_workers=nw,
                      pin_memory=True, drop_last=shuffle if drop_last is None else drop_last,
                      persistent_workers=nw > 0, prefetch_factor=4 if nw > 0 else None)

@torch.no_grad()
def fwd(img):
    with torch.autocast('cuda', dtype=AMP, enabled=dev == 'cuda'):
        return backbone(pixel_values=img).last_hidden_state[:, 1:].float()   # BO CLS

_b = next(iter(loader(I_VA[:8], False, False, 8, nw=0)))
N_TOK = {}
for a in ARMS:
    N_TOK[a] = fwd(_b['im_' + a].to(dev)).shape[1]
    exp = (ARM_IMG[a]['size'] // PATCH) ** 2
    assert N_TOK[a] == exp, f'{a}: {N_TOK[a]} token, cho {exp}'
print(f'cong 2: so token dung moi arm  ' + ' | '.join(f'{a}:{N_TOK[a]:,}' for a in ARMS))

# --- cong 3: BA ANH PHAI KHAC NHAU. Bat loi stage nham (ba thu muc tro cung mot bo anh). ---
_px = {a: _b['im_' + a] for a in ARMS}
for i, a in enumerate(ARMS):
    for b2 in ARMS[i+1:]:
        if _px[a].shape == _px[b2].shape:
            _dd = (_px[a] - _px[b2]).abs().max().item()
            assert _dd > 1e-3, (f'anh cua `{a}` va `{b2}` GIONG HET nhau (lech {_dd:.1e}) -> stage '
                                'nham, hai arm dang doc cung mot bo anh.')
_v = {a: float(_px[a].std()) for a in ARMS}
print(f'cong 3: ba bo anh KHAC nhau OK | do lech chuan pixel ' +
      ' '.join(f'{a}:{v:.3f}' for a, v in _v.items()))

stage ba bo anh:
  lb518     local       0 / 377,110


stage lb518:   0%|          | 0/378 [00:00<?, ?tar/s]

  lb518     stage xong 377,110 anh | 34.4 phut
  crop518   local       0 / 377,110


stage crop518:   0%|          | 0/378 [00:00<?, ?tar/s]

  crop518   stage xong 377,110 anh | 11.9 phut
  lb672     local       0 / 377,110


...

model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]


cong 1: backbone `pre` OK (lech so voi pretrain 0)
cong 2: so token dung moi arm  lb518:1,369 | crop518:1,369 | lb672:2,304
cong 3: ba bo anh KHAC nhau OK | do lech chuan pixel lb518:1.255 crop518:0.999 lb672:1.256


## C2 · Đầu đọc — `MLDecoder` verbatim `37`/`38*`/`45`, **một bản cho mỗi arm**

Ba arm dùng **đúng một lớp**, **đúng một siêu tham số**, khác nhau **duy nhất** ở chuỗi token đầu
vào. `MLDecoder` không có tham số nào phụ thuộc số token ⇒ ba arm **khớp tham số từng con số**, và
`C2` assert điều đó.

In [ ]:
class GroupFC(nn.Module):
    '''Moi lop mot vector trong so rieng -> 1 logit. Khong chia se giua cac lop.'''
    def __init__(s, C, d):
        super().__init__()
        s.w = nn.Parameter(torch.randn(C, d) * d ** -0.5); s.b = nn.Parameter(torch.zeros(C))
    def forward(s, h): return (h * s.w).sum(-1) + s.b

class XAttn(nn.Module):
    def __init__(s, d, nhead=8, ff=2048, drop=0.1):
        super().__init__()
        s.at = nn.MultiheadAttention(d, nhead, dropout=drop, batch_first=True)
        s.n1, s.n2 = nn.LayerNorm(d), nn.LayerNorm(d)
        s.ff = nn.Sequential(nn.Linear(d, ff), nn.GELU(), nn.Dropout(drop), nn.Linear(ff, d))
    def forward(s, q, kv, need_w=False):
        a, w = s.at(q, kv, kv, need_weights=need_w, average_attn_weights=True)
        h = s.n1(q + a)
        return s.n2(h + s.ff(h)), w

class MLDecoder(nn.Module):
    '''Q = C query lop (KHONG hoc, dung theo bai goc). K/V = TOAN BO token anh.'''
    def __init__(s, C, d_in, d=512, nhead=8, ff=1024, drop=0.1):
        super().__init__()
        s.q = nn.Embedding(C, d); s.q.weight.requires_grad_(False)
        s.pin = nn.Linear(d_in, d)
        s.img = XAttn(d, nhead, ff, drop)
        s.out = GroupFC(C, d)
    def forward(s, patch, need_w=False):
        q = s.q.weight.unsqueeze(0).expand(patch.shape[0], -1, -1)
        h, w = s.img(q, s.pin(patch), need_w)
        return s.out(h), w

class ASL(nn.Module):
    '''Verbatim nb11/12/13/15/35/37/38*/44*/45. gn=4, gp=0 la cau hinh 35 da chot.'''
    def __init__(s, gn=4, gp=0, clip=0.05, eps=1e-8):
        super().__init__(); s.gn, s.gp, s.clip, s.eps = gn, gp, clip, eps
    def forward(s, x, y):
        y = y.float(); xp = torch.sigmoid(x); xn = 1 - xp
        if s.clip: xn = (xn + s.clip).clamp(max=1)
        lo = y*torch.log(xp.clamp(min=s.eps)) + (1-y)*torch.log(xn.clamp(min=s.eps))
        pt = xp*y + xn*(1-y); g = s.gp*y + s.gn*(1-y)
        return -(lo * torch.pow(1-pt, g)).mean()

def build_arms(seed=SEED):
    '''Gieo lai seed TRUOC TUNG arm. Goi `manual_seed` mot lan roi dung ba arm tuan tu thi RNG
       chay tiep => ba arm khoi tao KHAC nhau, va mot phan hieu do duoc se la may man khoi tao
       chu khong phai anh. Voi thiet ke ghep cap thi do la confound; dry-run da bat dung loi nay.'''
    M = {}
    for a in ARMS:
        torch.manual_seed(seed)
        M[a] = MLDecoder(C, D_CXR, D_MLD)
    return nn.ModuleDict(M).to(dev)

MOD = build_arms()
_np_ = {a: sum(p.numel() for p in MOD[a].parameters() if p.requires_grad) for a in ARMS}
print(f'{"arm":<9} {"tham so":>12}')
for a in ARMS: print(f'  {a:<9} {_np_[a]:>12,}')

assert not any(p.requires_grad for p in backbone.parameters()), 'backbone chua dong cung'
assert not MOD[ARMS[0]].q.weight.requires_grad, 'query_embed dang hoc'
# --- cong 4: ba arm PHAI khop tham so TUNG CON SO. `MLDecoder` khong co tham so nao phu thuoc
#     so token, nen bat ky chenh lech nao la loi cau hinh. ---
assert len(set(_np_.values())) == 1, f'khop tham so HONG: {_np_}'
print(f'\ncong 4: ba arm khop tham so TUYET DOI ({list(_np_.values())[0]:,} moi arm)')

# --- cong 5: khoi tao PHAI giong het nhau giua ba arm (cung seed) => hieu do dung ANH, khong
#     phai do mot arm boc duoc khoi tao may man hon. ---
_r = {a: float(MOD[a].out.w.flatten()[:8].abs().sum()) for a in ARMS}
assert len(set(f'{v:.6f}' for v in _r.values())) == 1, f'khoi tao LECH giua cac arm: {_r}'
print(f'cong 5: ba arm khoi tao GIONG HET nhau OK')

arm            tham so
  lb518        2,517,032
  crop518      2,517,032
  lb672        2,517,032

cong 4: ba arm khop tham so TUYET DOI (2,517,032 moi arm)
cong 5: ba arm khoi tao GIONG HET nhau OK


/tmp/ipykernel_1118/2318679269.py:67: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  _r = {a: float(MOD[a].out.w.flatten()[:8].abs().sum()) for a in ARMS}


## C3 · Chấm điểm — `ap_cols` tự viết + gộp study, verbatim `45`

In [ ]:
from sklearn.metrics import average_precision_score
_sid_va = ORDER.study_id.to_numpy()[VA]
_uid, _inv = np.unique(_sid_va, return_inverse=True)
_isF = (V_.view_group.to_numpy()[VA] == 'frontal') & V_.use.to_numpy()[VA]
_isL = (V_.view_group.to_numpy()[VA] == 'lateral') & V_.use.to_numpy()[VA]

def pool_study(p):
    out = np.zeros((len(_uid), p.shape[1]), 'float64'); ws = np.zeros(len(_uid))
    for mask, w in ((_isF, W_FRONTAL), (_isL, W_LATERAL)):
        if not mask.any(): continue
        cnt = np.bincount(_inv[mask], minlength=len(_uid))
        acc = np.stack([np.bincount(_inv[mask], weights=p[mask, j].astype('float64'),
                                    minlength=len(_uid)) for j in range(p.shape[1])], 1)
        has = cnt > 0
        out[has] += w * acc[has] / cnt[has, None]; ws[has] += w
    ok = ws > 0; out[ok] /= ws[ok, None]; return out, ok
_first = np.zeros(len(_uid), int); _first[_inv[::-1]] = np.arange(VA.sum())[::-1]
_Ys = YM[VA][_first]

def ap_cols(Yc, Sc):
    o = np.full(Yc.shape[1], np.nan)
    for j in range(Yc.shape[1]):
        y, s = Yc[:, j], Sc[:, j]; n1 = y.sum()
        if n1 == 0 or n1 == len(y): continue
        r = np.argsort(-s, kind='stable'); yy, ss = y[r], s[r]
        tp, fp = np.cumsum(yy), np.cumsum(1 - yy)
        last = np.r_[np.diff(ss) != 0, True]; tp, fp = tp[last], fp[last]
        pr, rc = tp / (tp + fp), tp / n1
        o[j] = float((np.diff(np.r_[0.0, rc]) * pr).sum())
    return o
tiers = lambda ap: {t: float(np.nanmean(ap[i])) for t, i in TIER.items()}

# --- cong 6: cham lai doc lap bang sklearn tren mot lat nho, phai khop toi ~1e-8 (33 C4 dat 5e-9)
_rs6 = np.random.default_rng(6); _yy = (_rs6.random((3000, C)) < 0.2).astype('float32')
_ss = _rs6.random((3000, C)).astype('float32')
_a1 = ap_cols(_yy, _ss)
_a2 = np.array([average_precision_score(_yy[:, j], _ss[:, j]) for j in range(C)])
_dd = float(np.nanmax(np.abs(_a1 - _a2)))
assert _dd < 1e-6, f'ap_cols lech sklearn {_dd:.2e}'
print(f'cong 6: ap_cols khop sklearn toi {_dd:.2e}')
print(f'val {VA.sum():,} anh -> {len(_uid):,} study')

cong 6: ap_cols khop sklearn toi 1.69e-08
val 39,293 anh -> 23,800 study


## C4 · Huấn luyện — **ba arm trên MỘT vòng lặp**

Ba backbone-forward mỗi batch (không dùng chung được: ba ảnh khác nhau), ba đầu đọc, một `loss`
tổng. Tham số rời nhau ⇒ `d(L1+L2+L3)/dθ_i = dLi/dθ_i`, **tương đương tuyệt đối** với ba lần chạy
riêng.

`clip` gradient **riêng từng arm**; một arm cho logit không hữu hạn thì **bỏ arm đó ở bước đó**,
không giết cả lần chạy. Checkpoint **mỗi epoch** ⇒ đứt thì mất đúng một epoch.

In [ ]:
crit = ASL(gn=4, gp=0).to(dev)

@torch.no_grad()
def predict(M, idx, bs=48):
    M.eval()
    P = {a: np.zeros((len(idx), C), 'float32') for a in ARMS}
    for b in tqdm(loader(idx, False, False, bs, nw=0, drop_last=False), desc='eval', leave=False):
        for a in ARMS:
            lg, _ = M[a](fwd(b['im_' + a].to(dev, non_blocking=True)))
            P[a][b['pos'].numpy()] = torch.sigmoid(lg.float()).cpu().numpy()
    _bad = {a: int(np.isnan(v).sum()) for a, v in P.items() if np.isnan(v).any()}
    assert not _bad, f'predict cho ra NaN: {_bad}'
    for a in ARMS:
        assert np.abs(P[a]).sum(1).min() > 0, f'{a}: co hang khong duoc ghi -> `pos` sai'
    return P

def make_opt(M):
    ps = [p for p in M.parameters() if p.requires_grad]
    dec = [p for p in ps if p.ndim > 1]; nod = [p for p in ps if p.ndim <= 1]
    return torch.optim.AdamW([{'params': dec, 'weight_decay': WD},
                              {'params': nod, 'weight_decay': 0.0}], lr=LR)

P_LOG = os.path.join(RES51, 'epoch_log.json')
LOG = json.load(open(P_LOG)) if os.path.exists(P_LOG) else {}
DL  = loader(I_TF, True, True, BATCH)

for seed in SEEDS:
    sk = f's{seed}'
    if LOG.get(sk, {}).get('done'): print(f'{sk}: da xong, bo qua'); continue
    MOD = build_arms(seed)
    PARAM_OF = {a: [p for p in MOD[a].parameters() if p.requires_grad] for a in ARMS}
    N_CLIP = {a: 0 for a in ARMS}; N_SKIP = {a: 0 for a in ARMS}; N_STEP = 0
    opt = make_opt(MOD)
    steps = math.ceil(len(I_TF) / (BATCH * ACCUM)) * EPOCHS; warm = int(WARMUP * steps)
    sch = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: s/max(warm,1) if s < warm else
                                            0.5*(1+math.cos(math.pi*(s-warm)/max(steps-warm,1))))
    LOG.setdefault(sk, {})
    P_RES = os.path.join(CKPT, f'{sk}_resume.pt'); ep0 = 0
    if os.path.exists(P_RES):
        _r = torch.load(P_RES, map_location=dev, weights_only=False)
        MOD.load_state_dict(_r['mod']); opt.load_state_dict(_r['opt']); sch.load_state_dict(_r['sch'])
        N_CLIP, N_SKIP, N_STEP, ep0 = _r['n_clip'], _r['n_skip'], _r['n_step'], _r['ep'] + 1
        print(f'{sk}: resume tu epoch {ep0+1}/{EPOCHS}')

    for ep in range(ep0, EPOCHS):
        MOD.train()
        t0, tot, nb, n_bad = time.time(), {a: 0.0 for a in ARMS}, 0, 0
        opt.zero_grad(set_to_none=True)
        for j, b in enumerate(tqdm(DL, desc=f'{sk} ep{ep+1}/{EPOCHS}', leave=False)):
            y = b['y'].to(dev); loss, bad = 0.0, []
            for a in ARMS:
                lg, _ = MOD[a](fwd(b['im_' + a].to(dev, non_blocking=True)))
                if not torch.isfinite(lg).all():
                    bad.append(a); N_SKIP[a] += 1; continue
                l_ = crit(lg.float(), y); loss = loss + l_; tot[a] += float(l_.detach())
            if bad:
                n_bad += 1
                if n_bad <= 3 or n_bad % 200 == 0:
                    _mx = {a: max(float(p.abs().max()) for p in PARAM_OF[a]) for a in bad}
                    print(f'   [{sk} ep{ep+1} buoc {j}] logit khong huu han: {bad} | '
                          + ' '.join(f'{a}:|w|max {v:.3g}' for a, v in _mx.items()))
            if not torch.is_tensor(loss):
                opt.zero_grad(set_to_none=True); continue
            (loss / ACCUM).backward(); nb += 1
            if (j + 1) % ACCUM == 0:
                # Kiem gradient TRUOC khi buoc: clip_grad_norm_ voi total_norm = nan cho clip_coef
                # = nan va BOI BAN toan bo tham so -> mot buoc hong lam chet vinh vien.
                for a, ps in PARAM_OF.items():
                    gn = torch.nn.utils.clip_grad_norm_(ps, CLIP, error_if_nonfinite=False)
                    if not torch.isfinite(gn):
                        for p in ps:
                            if p.grad is not None: p.grad.zero_()
                        N_SKIP[a] += 1
                    elif float(gn) > CLIP: N_CLIP[a] += 1
                N_STEP += 1
                opt.step(); sch.step(); opt.zero_grad(set_to_none=True)

        PV = predict(MOD, I_VA); PS = predict(MOD, I_SL)
        ensure_res()
        for a, p in PV.items(): np.save(os.path.join(RES51, f'p_{a}_{sk}_ep{ep+1}.npy'), p)
        _e = {}
        for a, p in PV.items():
            ap = ap_cols(YM[VA], p)
            _e[a] = dict(mAP_anh=float(np.nanmean(ap)), **tiers(ap),
                         sel=float(np.nanmean(ap_cols(YM[SL], PS[a]))))
        LOG[sk][str(ep)] = _e; json.dump(LOG, open(P_LOG, 'w'))
        torch.save(dict(mod=MOD.state_dict(), opt=opt.state_dict(), sch=sch.state_dict(),
                        n_clip=N_CLIP, n_skip=N_SKIP, n_step=N_STEP, ep=ep, seed=seed), P_RES)
        # `loss` in TRUNG BINH TUNG ARM, khong phai tong — tong lam hai lan chay khac so arm
        # trong nhu khac nhau (loi hien thi con ton tu 38c).
        print(f'{sk} ep{ep+1:>2} | {nb*BATCH/max(time.time()-t0,1e-9):.0f} img/s '
              f'| {(time.time()-t0)/60:.1f} ph | loss ' +
              ' '.join(f'{a}:{tot[a]/max(nb,1):.4f}' for a in ARMS))
        print('   VAL      ' + ' '.join(f'{a:>9}:{_e[a]["mAP_anh"]:.4f}' for a in ARMS))
        print('   LAT CHON ' + ' '.join(f'{a:>9}:{_e[a]["sel"]:.4f}' for a in ARMS))
        print('   ^ CHON `E` BANG DONG `LAT CHON`, KHONG bang dong `VAL`')
        _d1 = _e['lb672']['mAP_anh']   - _e['lb518']['mAP_anh']
        _d2 = _e['crop518']['mAP_anh'] - _e['lb518']['mAP_anh']
        _d3 = _e['crop518']['mAP_anh'] - _e['lb672']['mAP_anh']
        print(f'   lb672   - lb518 = {_d1:+.4f}  <- DO PHAN GIAI thuan (49 o mean-pool: -0.0010)')
        print(f'   crop518 - lb518 = {_d2:+.4f}  (49 o mean-pool: +0.0082 — 60% cua no la PHA LOANG)')
        print(f'   crop518 - lb672 = {_d3:+.4f}  <- gia cua FOV o cung 0.626 mm/px')
        _f = {a: N_CLIP[a]/max(N_STEP,1) for a in ARMS}
        if max(_f.values()) > 0.01:
            print('   clip kich hoat: ' + ' '.join(f'{a}:{v:.0%}' for a, v in _f.items() if v > 0.01))
        if max(N_SKIP.values()) > 0:
            print('   BUOC BI BO: ' + ' '.join(f'{a}:{v}' for a, v in N_SKIP.items() if v > 0)
                  + '  -> arm nao bi bo nhieu thi con so cua no KHONG doc duoc')
    ensure_res()
    torch.save(dict(mod=MOD.state_dict(), seed=seed, n_skip=N_SKIP),
               os.path.join(CKPT, f'{sk}.pt'))
    LOG[sk]['done'] = True; LOG[sk]['n_skip'] = N_SKIP; json.dump(LOG, open(P_LOG, 'w'))
    print(f'{sk}: xong' + (f' | buoc bi bo: {N_SKIP}' if max(N_SKIP.values()) else ''))
print('tat ca seed xong')

s42 ep1/10:   0%|          | 0/3434 [00:00<?, ?it/s]

eval:   0%|          | 0/819 [00:00<?, ?it/s]

eval:   0%|          | 0/815 [00:00<?, ?it/s]

s42 ep 1 | 76 img/s | 48.0 ph | loss lb518:0.0444 crop518:0.0444 lb672:0.0445
   VAL          lb518:0.2159   crop518:0.2150     lb672:0.2154
   LAT CHON     lb518:0.2052   crop518:0.2054     lb672:0.2046
   ^ CHON `E` BANG DONG `LAT CHON`, KHONG bang dong `VAL`
   lb672   - lb518 = -0.0005  <- DO PHAN GIAI thuan (49 o mean-pool: -0.0010)
   crop518 - lb518 = -0.0009  (49 o mean-pool: +0.0082 — 60% cua no la PHA LOANG)
   crop518 - lb672 = -0.0004  <- gia cua FOV o cung 0.626 mm/px


s42 ep2/10:   0%|          | 0/3434 [00:00<?, ?it/s]

...

s42 ep 9 | 81 img/s | 45.3 ph | loss lb518:0.0382 crop518:0.0383 lb672:0.0382
   VAL          lb518:0.2395   crop518:0.2368     lb672:0.2367
   LAT CHON     lb518:0.2279   crop518:0.2255     lb672:0.2278
   ^ CHON `E` BANG DONG `LAT CHON`, KHONG bang dong `VAL`
   lb672   - lb518 = -0.0027  <- DO PHAN GIAI thuan (49 o mean-pool: -0.0010)
   crop518 - lb518 = -0.0026  (49 o mean-pool: +0.0082 — 60% cua no la PHA LOANG)
   crop518 - lb672 = +0.0001  <- gia cua FOV o cung 0.626 mm/px


s42 ep10/10:   0%|          | 0/3434 [00:00<?, ?it/s]

eval:   0%|          | 0/819 [00:00<?, ?it/s]

eval:   0%|          | 0/815 [00:00<?, ?it/s]

s42 ep10 | 81 img/s | 45.4 ph | loss lb518:0.0379 crop518:0.0380 lb672:0.0379
   VAL          lb518:0.2392   crop518:0.2365     lb672:0.2364
   LAT CHON     lb518:0.2276   crop518:0.2252     lb672:0.2276
   ^ CHON `E` BANG DONG `LAT CHON`, KHONG bang dong `VAL`
   lb672   - lb518 = -0.0028  <- DO PHAN GIAI thuan (49 o mean-pool: -0.0010)
   crop518 - lb518 = -0.0027  (49 o mean-pool: +0.0082 — 60% cua no la PHA LOANG)
   crop518 - lb672 = +0.0001  <- gia cua FOV o cung 0.626 mm/px
s42: xong
tat ca seed xong


## C5 · Ô 2×2 + **kiểm giả định trung tâm**

Giả định của cả lập luận: *attention tự hạ trọng số token đệm đen ⇒ thành phần **pha loãng** của
`49` phải biến mất*. `49` đo `nosq518 − lb518 = +0.0049` — đúng thành phần ấy, và nó chiếm **60%**
của `crop518 − lb518 = +0.0082`.

Ở đây kiểm gián tiếp: nếu attention trung hoà pha loãng thì `crop518 − lb518` phải **co lại rõ**
so với `+0.0082`. Không co ⇒ giả định sai, và mọi diễn giải phía trên phải viết lại.

In [ ]:
LOG = json.load(open(P_LOG)); EPk = str(EPOCHS - 1)
rows = [dict(seed=s, arm=a, **v) for s in SEEDS if EPk in LOG.get(f's{s}', {})
        for a, v in LOG[f's{s}'][EPk].items()]
R = pd.DataFrame(rows); assert len(R), 'chua co ket qua nao'
SD_OK = [s for s in SEEDS if EPk in LOG.get(f's{s}', {})]
for s in SEEDS:
    if s not in SD_OK: print(f's{s}: chua co ep{EPOCHS}')

print('=== TUYET DOI (TB qua seed) ===')
piv = R.pivot_table(index='arm', values=['mAP_anh','head','medium','tail','sel'],
                    aggfunc='mean').reindex(ARMS)
piv['bien_do'] = R.groupby('arm').mAP_anh.agg(lambda s: s.max()-s.min()).reindex(ARMS)
for c in ('mm_px','fov'): piv[c] = [ARM_IMG[a]['mm_px' if c=='mm_px' else 'fov'] for a in ARMS]
piv['token'] = [N_TOK[a] for a in ARMS]
piv['moc49'] = [MOC49[a] for a in ARMS]
print(piv.to_string(float_format=lambda v: f'{v:.4f}'))
NOISE = float(np.nanmax(piv.bien_do)) if len(SD_OK) > 1 else float('nan')
print(f'\nbien do seed: ' + (f'{NOISE:.4f}' if len(SD_OK) > 1 else
      'CHUA DOC DUOC (1 seed). Dung sign-consistency qua epoch o duoi thay the.'))

g = lambda a, s: float(R[(R.arm == a) & (R.seed == s)].mAP_anh.iloc[0])
def d(a, b, ngh, moc):
    v = [g(a, s) - g(b, s) for s in SD_OK]
    ok = all(x > 0 for x in v) or all(x < 0 for x in v)
    bd = (max(v) - min(v)) if len(v) > 1 else float('nan')
    dat = '---' if len(v) < 2 else ('DAT' if (ok and abs(np.mean(v)) > bd) else '---')
    print(f'  {a:>8} - {b:<8} {np.mean(v):+.4f}  (bien do {bd:.4f}, {dat})  '
          f'moc mean-pool 49: {moc:+.4f}   {ngh}')

print('\n=== O 2x2 ===')
d('lb672', 'lb518',   'DO PHAN GIAI thuan (FOV 100%, dem 18%, chi doi mm/px)', MOC49['lb672']-MOC49['lb518'])
d('crop518', 'lb518', 'mua do phan giai BANG FOV (cung 1,369 token, cung FLOPs)', MOC49['crop518']-MOC49['lb518'])
d('crop518', 'lb672', 'gia cua FOV o cung 0.626 mm/px', MOC49['crop518']-MOC49['lb672'])

print('')
print('=== KIEM GIA DINH TRUNG TAM: attention co trung hoa PHA LOANG khong? ===')
_c = float(np.mean([g('crop518', s) - g('lb518', s) for s in SD_OK]))
_m = MOC49['crop518'] - MOC49['lb518']
print(f'  crop518 - lb518   mean-pool (49) {_m:+.4f}  ->  ML-Decoder {_c:+.4f}   '
      f'co lai {(_m-_c):+.4f}')
print(f'  49 do rieng thanh phan pha loang: nosq518 - lb518 = +0.0049 (60% cua {_m:+.4f})')
if _c < _m - 0.003:
    print('  => CO co lai dang ke. Khop du doan: attention tu ha trong so token dem den,')
    print('     nen phan mAP mua duoc bang "bot token rong" khong con.')
elif _c > _m - 0.001:
    print('  => KHONG co lai. Gia dinh SAI — hoac attention khong bo qua token rong duoc,')
    print('     hoac +0.0082 cua 49 khong phai do pha loang. Phai viet lai dien giai o C7.')
else:
    print('  => vung xam, khong ket luan.')

print('')
print('=== ON DINH QUA EPOCH (thay cho bien do seed khi chi co 1 seed) ===')
for a, b in (('lb672','lb518'), ('crop518','lb518'), ('crop518','lb672')):
    v = [LOG[f's{SD_OK[0]}'][str(e)][a]['mAP_anh'] - LOG[f's{SD_OK[0]}'][str(e)][b]['mAP_anh']
         for e in range(EPOCHS) if str(e) in LOG[f's{SD_OK[0]}']]
    npos = sum(x > 0 for x in v)
    print(f'  {a:>8} - {b:<8} duong {npos}/{len(v)} epoch | 5 epoch cuoi ' +
          ' '.join(f'{x:+.4f}' for x in v[-5:]))
print('  (45 dung dung tieu chi nay: `G_t1prior` duong 10/10 epoch)')
ensure_res(); R.to_csv(os.path.join(RES51, 'main_table.csv'), index=False)

=== TUYET DOI (TB qua seed) ===
          head  mAP_anh  medium    sel   tail  bien_do  mm_px    fov  token  moc49
arm                                                                               
lb518   0.4482   0.2392  0.1679 0.2276 0.0959   0.0000 0.8120 1.0000   1369 0.2044
crop518 0.4486   0.2365  0.1658 0.2252 0.0887   0.0000 0.6260 0.7150   1369 0.2126
lb672   0.4495   0.2364  0.1670 0.2276 0.0857   0.0000 0.6260 1.0000   2304 0.2034

bien do seed: CHUA DOC DUOC (1 seed). Dung sign-consistency qua epoch o duoi thay the.

=== O 2x2 ===
     lb672 - lb518    -0.0028  (bien do nan, ---)  moc mean-pool 49: -0.0010   DO PHAN GIAI thuan (FOV 100%, dem 18%, chi doi mm/px)
   crop518 - lb518    -0.0027  (bien do nan, ---)  moc mean-pool 49: +0.0082   mua do phan giai BANG FOV (cung 1,369 token, cung FLOPs)
   crop518 - lb672    +0.0001  (bien do nan, ---)  moc mean-pool 49: +0.0092   gia cua FOV o cung 0.626 mm/px

=== KIEM GIA DINH TRUNG TAM: attention co trung hoa PHA LOANG khong? =

## C6 · Nhóm nhãn — FOV có đắt hơn dưới attention không?

`49` ở `mean(patch)`: `crop518` thắng **cả** ở nhóm ngoại vi (`0.2098`, cao nhất) — token ngoại vi
vốn đã bị pha loãng nên mất chúng gần như miễn phí.

Dưới attention thì khác: token ngoại vi **dùng được khi cần**. Nếu FOV thật sự có giá thì `crop518`
phải **thua ở nhóm ngoại vi** ngay cả khi thắng macro. Đây là phép thử **cơ chế**, không phải một
con số mAP nữa.

In [ ]:
PER = [l for l in LAB40 if re.search(
    r'rib fracture|clavicle|subcutaneous|scoliosis|pneumomediastinum|pleural thickening',
    l, re.I)]
IP = np.array([LAB40.index(l) for l in PER], dtype=int)
IC = np.setdiff1d(np.arange(C), IP)
print(f'nhom NGOAI VI / DINH PHOI ({len(PER)}): {PER}')
print('  (them Pneumomediastinum va Pleural Thickening: o 49 chung la HAI nhan duy nhat crop518')
print('   thua — khi trung that lan len co, va mang phoi sat thanh nguc)\n')

sk = f's{SD_OK[0]}'
AP = {}
for a in ARMS:
    f = os.path.join(RES51, f'p_{a}_{sk}_ep{EPOCHS}.npy')
    if os.path.exists(f): AP[a] = ap_cols(YM[VA], np.load(f))
print(f'{"arm":<9} {"macro":>8} {"NGOAI VI":>9} {"trung tam":>10} {"tail":>8}')
for a in ARMS:
    if a not in AP: continue
    ap = AP[a]
    print(f'{a:<9} {np.nanmean(ap):>8.4f} {np.nanmean(ap[IP]):>9.4f} '
          f'{np.nanmean(ap[IC]):>10.4f} {np.nanmean(ap[TIER["tail"]]):>8.4f}')
if 'crop518' in AP and 'lb518' in AP:
    _pp = np.nanmean(AP['crop518'][IP]) - np.nanmean(AP['lb518'][IP])
    _cc = np.nanmean(AP['crop518'][IC]) - np.nanmean(AP['lb518'][IC])
    print(f'\n  crop518 - lb518:  NGOAI VI {_pp:+.4f}  |  trung tam {_cc:+.4f}')
    print(f'  49 o mean-pool:   NGOAI VI +0.0088  |  trung tam +0.0082  (deu nhau -> FOV mien phi)')
    if _pp < _cc - 0.003:
        print('  => Duoi attention, FOV CO GIA: crop lam hong dung nhom nhan can vung ngoai vi.')
    else:
        print('  => FOV van gan nhu mien phi. Vung 28.5% bi cat khong mang nhan nao dang ke,')
        print('     KE CA khi dau doc du suc dung no.')

print('\n=== HIEU TUNG NHAN: TAP TRUNG hay NANG DEU ===')
for a in ('lb672', 'crop518'):
    if a not in AP or 'lb518' not in AP: continue
    dd = AP[a] - AP['lb518']; o = np.argsort(-dd)
    print(f'\n{a} - lb518: duong {int((dd>0).sum())}/{C} | TB {np.nanmean(dd):+.4f} | '
          f'trung vi {np.nanmedian(dd):+.4f}')
    print(f'  {"loi nhat":<30}{"":4}{"thiet nhat"}')
    for k in range(6):
        hi, lo = o[k], o[-1-k]
        print(f'    {LAB40[hi]:<28}{dd[hi]:+.4f}    {LAB40[lo]:<28}{dd[lo]:+.4f}')
print('\n  TAP TRUNG (TB >> trung vi, dinh co ly do giai phau) -> hieu ung ANH that')
print('  NANG DEU  (gan 40/40, TB ~ trung vi)                 -> nghi khac biet DUONG ONG')

nhom NGOAI VI / DINH PHOI (5): ['Clavicle Fracture', 'Pleural Thickening', 'Pneumomediastinum', 'Rib Fracture', 'Subcutaneous Emphysema']
  (them Pneumomediastinum va Pleural Thickening: o 49 chung la HAI nhan duy nhat crop518
   thua — khi trung that lan len co, va mang phoi sat thanh nguc)

arm          macro  NGOAI VI  trung tam     tail
lb518       0.2392    0.1843     0.2471   0.0959
crop518     0.2365    0.1739     0.2455   0.0887
lb672       0.2364    0.1798     0.2445   0.0857

  crop518 - lb518:  NGOAI VI -0.0104  |  trung tam -0.0016
  49 o mean-pool:   NGOAI VI +0.0088  |  trung tam +0.0082  (deu nhau -> FOV mien phi)
  => Duoi attention, FOV CO GIA: crop lam hong dung nhom nhan can vung ngoai vi.

=== HIEU TUNG NHAN: TAP TRUNG hay NANG DEU ===

lb672 - lb518: duong 19/40 | TB -0.0028 | trung vi -0.0005
  loi nhat                          thiet nhat
    Granuloma                   +0.0280    Azygos Lobe                 -0.0531
    Hydropneumothorax           +0.0125    Pneum

## C7 · Ghi `REVIEW_RESOLUTION_MLDEC.md`

In [ ]:
L = []; w = L.append
w('# REVIEW · độ phân giải dưới ML-Decoder (`nb51`)'); w('')
w(f'`{RUN_TAG}` · backbone **{BB_MODE}** · `asl40` · `E={EPOCHS}` · **epoch cuối** · '
  f'seeds {list(SD_OK)} · val {int(VA.sum()):,} ảnh · ba arm **một vòng lặp, ghép cặp tuyệt đối**.')
w('')
w('| arm | px | token | mm/px | FOV | đệm | mAP ảnh | tail | lát chọn | mốc `49` (mean-pool) |')
w('|---|---|---|---|---|---|---|---|---|---|')
for a in ARMS:
    if a not in piv.index: continue
    r = piv.loc[a]; c = ARM_IMG[a]
    w(f'| `{a}` | {c["size"]} | {N_TOK[a]:,} | {c["mm_px"]:.3f} | {c["fov"]:.1%} | {c["pad"]:.0%} | '
      f'{r["mAP_anh"]:.4f} | {r["tail"]:.4f} | {r["sel"]:.4f} | {MOC49[a]:.4f} |')
w(''); w('```')
for a, b in (('lb672','lb518'), ('crop518','lb518'), ('crop518','lb672')):
    v = [g(a, s) - g(b, s) for s in SD_OK]
    w(f'{a:>8} - {b:<8} {np.mean(v):+.4f}   mean-pool (49): {MOC49[a]-MOC49[b]:+.4f}')
w('```'); w('')
w('## Những gì kết quả này KHÔNG trả lời'); w('')
w(f'- **Backbone `{BB_MODE}`.** ' + (
    'Pretrain nên không thiên vị arm nào — nhưng cũng chưa cho biết một backbone **fine-tune trên '
    'chính arm đó** sẽ xếp hạng thế nào. Arm thắng ở đây cần một lần `38a` (~6 h) để thu về ở mức '
    '`ft`.' if BB_MODE == 'pre' else
    '`38a` fine-tune trên `lb518`, nên `crop518`/`lb672` chịu **lệch miền**. Kết quả **âm** của hai '
    'arm đó KHÔNG đọc được; chỉ kết quả dương mới là cận dưới hợp lệ.'))
w('- **Ô 2×2 vẫn khuyết cell `(0.812 mm/px, FOV 71.5%)`** ⇒ chưa kiểm được **tính cộng tính** của '
  'hai hiệu ứng.')
w('- **`lb672` tốn `1.95×` GPU** so với `518` (2,304 vs 1,369 token; `L·(12·N·d² + 2·N²·d)` cho '
  '294 vs 151 GMAC/ảnh). `crop518` cho **cùng `0.626 mm/px`** ở **đúng chi phí của `lb518`** — nó '
  'trả bằng FOV thay vì bằng compute.')
if len(SD_OK) < 2:
    w(f'- **Chỉ {len(SD_OK)} seed** ⇒ biên độ seed chưa đo được. Thay bằng tính nhất quán dấu qua '
      f'{EPOCHS} epoch (đúng tiêu chí `45` đã dùng cho `G_t1prior`), yếu hơn một bậc. Thêm seed 43 '
      'thì chỉ cần đổi `SEEDS` rồi chạy lại `C4` — resume ở mức epoch.')
w('- **Không có EHR.** `G_ehr = +0.007` là số hạng cộng thêm, đo ở 4 notebook khác; notebook này '
  'không nói gì về việc độ phân giải có **tương tác** với nhánh EHR hay không.')
w('- **Đích.** Nhãn CXR-LT do NLP trích từ **chính report của phim đó**. Không con số nào ở đây '
  'nói gì về tiên lượng, tử vong, chuyển ICU, tái nhập viện.')
TXT = '\n'.join(L)
ensure_res()
open(os.path.join(RES51, 'REVIEW_RESOLUTION_MLDEC.md'), 'w', encoding='utf-8').write(TXT)
print(TXT)

# REVIEW · độ phân giải dưới ML-Decoder (`nb51`)

`r1_pre` · backbone **pre** · `asl40` · `E=10` · **epoch cuối** · seeds [42] · val 39,293 ảnh · ba arm **một vòng lặp, ghép cặp tuyệt đối**.

| arm | px | token | mm/px | FOV | đệm | mAP ảnh | tail | lát chọn | mốc `49` (mean-pool) |
|---|---|---|---|---|---|---|---|---|---|
| `lb518` | 518 | 1,369 | 0.812 | 100.0% | 18% | 0.2392 | 0.0959 | 0.2276 | 0.2044 |
| `crop518` | 518 | 1,369 | 0.626 | 71.5% | 0% | 0.2365 | 0.0887 | 0.2252 | 0.2126 |
| `lb672` | 672 | 2,304 | 0.626 | 100.0% | 18% | 0.2364 | 0.0857 | 0.2276 | 0.2034 |

```
   lb672 - lb518    -0.0028   mean-pool (49): -0.0010
 crop518 - lb518    -0.0027   mean-pool (49): +0.0082
 crop518 - lb672    +0.0001   mean-pool (49): +0.0092
```

## Những gì kết quả này KHÔNG trả lời

- **Backbone `pre`.** Pretrain nên không thiên vị arm nào — nhưng cũng chưa cho biết một backbone **fine-tune trên chính arm đó** sẽ xếp hạng thế nào. Arm thắng ở đây cần một lần `38a` (~6 h) để thu về ở mức 